In [ ]:
print('TEST')

In [ ]:
## 이 노트북은 RAG 테스트 용도 입니다.
## 일반적으로 웹에서 사용하는 용도가 아닌, 프로그램을 만들어서 LLM모델을 사용하려고 한다
## LLM Model: Gemini(google) - 무료
## 컨셉: 메뉴데이터를 벡터데이터베이스 구축 한 후 사용자 질의에 응답하는 챗봇

In [5]:
# LLM 모델 테스트
# 우리가 사용할 LLM 모델은 Gemini
# 그래서 구글 AI STUDIO에서 키를 발급 받아야한다. 이 키를 api key
# 그리고 이 키를 로컬에서 관리한다. .env 폴더를 먼저 만든다 위치 꼭 기억 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()   #.env에 있는 구글 api 키를 자동으로 로딩

# 모델을 선정하고 파라미터 값을 설정할 수 있다.
llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash", 
                             temperature= 0.3,
                             max_output_tokens = 1024 # 대답에 대한 토큰 수 설정. 토큰을 판단하는 기준.
                             )
                             # 0.0: 일관성 높은 답변, 0.3 : 0.0보다는 창의성 추가, 일반적인 답변 
                             # 1.0: 기본값 - 창의적인 답변.. 1.5: 정말 창의적인 답변
                             # 창의적인 것을 llm은 단어와 의도를 보고 상황마다 답변이 달라 질 수 있음

# response = llm.invoke('대한민국 소개해 줘')
# print(response.content)

# 모델이 정상 동작임을 확인함.. RAG에서 G부분에서 사용할 모델 테스트 완료 됨.

# 연규할 항목 : 모델별로 성능 차이가 있다.. 개발 주제와 가장 적당한 모델을 찾는 것..
# 주제는 이렇고, 데이터는 이렇고. 그래서 open ai 사의 gpt X, google Gemini 3.5x버전 등을 사용
## 어떤 모델의 토큰량과 정답도를 고려하여 최종 000모델로 선택한다.

In [3]:
## RAG 처럼 테스트
query = "짜장면은 달고 짜면서 맛있어. 짬뽕은 얼큰하고 맛있어.. 이게 내가 주는 데이터야.. 이 데이터로 광고문구 만들어줘"
response = llm.invoke(query)
print(response.content)

[{'type': 'text', 'text': '제공해주신 데이터(**짜장면: 달고 짭조름함 / 짬뽕: 얼큰함 / 둘 다 맛있음**)의 핵심을 살려, 상황과', 'extras': {'signature': 'EoIbCv8aAWkUfRO9uKCO7yauWnppokIvNVSetXGuxc6zK6mIGmqUPFqlKOYAxGYRc0AMVEsUYG6/DeAIkDT9gSM2vJFLt5XoFyhUQt7Xp99ZMNqpdo0fN3JnYimhKd3+RyvriZe7cunIXry0iEjXlCys16wrvyKEXuC4AEGimkKq7OSmtYsTxi3t/DtFBwUvG+SSigxEwIRUcuVIvYLo9PMnWZUFfSlk4aZb3U24hZLLBeFAbb7YYSv5wq+DhA94w/D4RaM3npeXQVPb8KkAxMUBLO/lPZlD8eufqty17vYh3OUZ/WCaEDorS2lvQf3qIpV85M0mNdYUI4LG4vXV7pNVEFNTTyE5aSGiQ5pv7+abROoCKaJdJA9yV6p/itE/wQZuZNN3DbBHH818lT2PI8pYVZVdGB8OdUZHNzam/Q7Tm72PbR2AxifOIcUjKVut2I03I+Z7jz3tG7rXhhCaoBO70NfzcEKezo+qtnsVbGTOscSUtCVsLNqUu3D66KGEzNtf7BjStr0VBMBO+Jnv7B2ma2cSiHY4v849IojjyI5j72NnaIVYmZqUn0KMxqn+XucxkXG12aeZFoAkH+2PhDuO10QIqOwf582iVmru9itPL4pv+6hzuymwOTO2UYy9eY0HZqXDAEuzMsLhHqw8A+RNn95YyNRI5CaG+fl4cCM6y+/AFk83EKmsdyjJ2jlW1BCcjSgenE/l6ROPLoKc5GDxITi27+ES2W+x30csCGmXfFwsVEfAiTBg7+zWXI+pknF9kwrKWCF2k3D4svBWLrBwZ9o2JuF65IiLTmjw7Q06l+9EvJ0mBqWJd4u7dm9CXCxq4o4zXvp9JmocPMmqbKd5goyumWwT1+GwKZzsdWF0oOEshA9u1

In [7]:
# 1. LLM에게 전달 할때 우리가게(음식가게)의 메뉴의 정보를 제공해야 합니다
## 그러기 위해서 ..메뉴의 정보를 불러오고 > 벡터데이터로 구축해야한다.
## 우리가게 정보는 menus_recommendation.json에 있음
import json
from langchain_core.documents import Document   # 벡터디비 만들때는 데이터를 Document로 만들어야함
# 벡터디비 만들때는 데이터(우리는 json)를 Document로 만들어야함
# Document 객체는 키의 이름이 page_content와 metadata로 되어있음 (단순 암기 영역)

# 우리의 json 데이터 로드
with open("../storage/menus_recommendation.json", "r", encoding="utf-8") as file:
    data = json.load(file)
# print(data)

# 일부만 확인해보자
#print(data[:1])

# 데이터를 가져왔으니.. 벡터데이터베이스를 만들기 전에 Documents 문서 객체로 변환
# Document생성
docs = []   # document 문서를 저장할 배열
for row in data:
    #content 변수는 문자열이다
    content = f"""
        이름 : {row['name']}
        설명 : {row['description']}
    """
    # print(content)
    docmake = Document(
        page_content=content.strip(),
        metadata={
            "name": row['name'],
            "calories": int(row['calories']),
            "recommendation": int(row['recommendation'])
        }
    )
    docs.append(docmake)
print(docs[:2]) # 리스트에는 document 타입의 객체가 저장되어 있음 

[Document(metadata={'name': '마파두부', 'calories': 400, 'recommendation': 92}, page_content='이름 : 마파두부\n        설명 : 두부와 다진 고기를 매콤한 소스로 볶은 중국 요리'), Document(metadata={'name': '깐풍기', 'calories': 750, 'recommendation': 313}, page_content='이름 : 깐풍기\n        설명 : 튀긴 닭고기를 매콤달콤한 소스에 볶은 요리')]


In [8]:
# 위에서 생성한 documnet가 저장된 docs리스트로 벡터 데이터베이스를 생성해야 한다
# 이때 docs문장에 있는 한글을 숫자화 해야 한다
# 가장 좋은 성능을 보이는 트랜스포머로 만든 모델을 사용할 것이다.
# 한글을 벡터화 시키는 것을 임베딩모델이라고 하고, 우리가 사용할 임베딩 모델은
# 트랜스포머 방식으로 개발된 모델이다
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},  # GPU 있으면 "cuda"
    encode_kwargs={"normalize_embeddings": True} # 벡터 비교에서 중요한 건 “방향”임
)


C:\Users\human-11\AppData\Local\Temp\ipykernel_23140\4216328651.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
C:\Users\human-11\AppData\Local\Temp\ipykernel_23140\4216328651.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
d:\KJY\menus_project_rag\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://

In [10]:
## 벡터데이터베이스를 구축한다
## RAG에서 사용하는 벡터데이터베이스는 정형데이터베이스랑 다르다
## 종류는 pincorn faiss chroma 디비 등이 있다. 나는 chroma db를 사용할것이다
## 그래서 디비가 저장된 디렉토리를 생성함 .. chroma_db


## 벡터 데이터베이스 생성
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=docs,     ## 벡터 데이터 베이스를 구축할 정보
    embedding=embeddings,   ## docs의 문장을 해석하여 데이터베이스를 구축할 임베딩 객체
    persist_directory="../chroma_db/menus_db_meta" ## 데이터베이스 위치
)

In [9]:
## 벡터 데이터베이스 구축된 정보 확인
print("저장된 문서 개수:" , vectorstore._collection.count())

저장된 문서 개수: 97


In [11]:
## 벡터데이터베이스를 한 번 구축하면 다음에는 구축 안해도 되고 읽어 오기만 하면 된다
## 이미 구축된 데이터베이스에서 데이터를 읽어 오는 법
vectorstore = Chroma(
    persist_directory="../chroma_db/menus_db_meta",
    embedding_function=embeddings   # 주의할 점.. 벡터디비 구축 시 embedding 모델과
                                    # 읽어 올 때의 모델은 동일해야 한다
)

In [22]:
## 지금까지 작업으로 선정한 데이터(json)으로 벡터데이터베이스를 구축함
## 사용자 질의로 벡터데이터베이스와 유사도가 높은 문서로 검색해보자

## 사용자질문
query = "칼로리가 500이하인 튀겨서 만든 요리 추천해줘"
results = vectorstore.similarity_search(query, k=4)  

for doc in results:
    print(doc.page_content)

이름 : 치킨
        설명 : 닭고기를 바삭하게 튀기거나 구워 만든 음식
이름 : 치킨
        설명 : 닭고기를 바삭하게 튀기거나 구워 만든 음식
이름 : 깐풍기
        설명 : 튀긴 닭고기를 매콤달콤한 소스에 볶은 요리
이름 : 깐풍기
        설명 : 튀긴 닭고기를 매콤달콤한 소스에 볶은 요리


In [ ]:
## 파이프라인 구성
## 사용자가 질문을 함.. 이 질문을 LLM에게 분석을 시킨다.. 얻고 싶은 데이터는 질문, 칼로리, 추천수 3개의 정보를 얻어냄
## 질문과 칼로리와 추천수의 정보들로 이때 유사도 검색을 함..
## 유사도 검색 후 증강 문맥 생성은 동일함
## 중간에 질의문을 분석하도록 LLM을 사용하고 LLM이 분석한 결과(질문, 칼로리,추천수)로 유사도 검색을 함 > 증강 > 문맥생성(LLM)

In [ ]:
# 여기서 사용하는 LLM의 목적은 질문의 의도를 분석하려고 한다
# 질문을 분석해서 원하는 질문, 칼로리, 추천수를 먼저 분석하고 유사도 검색에 활용
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def analyze_query(user_query):
    analyze_promt = ChatPromptTemplate.from_template("""
        사용자의 질문을 분석하여 아래 json형식으로 결과를 출력해줘(다른 설명은 절대 붙이지 말 것)
        - query : 벡터검색을 이용한 핵심 키워드 또는 메뉴이름 (예:"매운요리","마라탕")
        - max_calories : 지문에 언급된 최대 칼로리 조건(숫자만 추출, 없으면 null)
        - min_recommendation : 질문에 언급된 최소 추천수 조건(숫자만 추출, 없으면 null)

        [사용자 질문] : {query}
        [출력형식(json)] : 
        {{"query":"추출된 검색어", "max_calories":null, "min_recommendation":null}}
    """)

    chain = analyze_promt | llm | StrOutputParser()
    result_str = chain.invoke({"query":user_query})

    try:
        return json.loads(result_str)   ## result_str값을 json으로 변경할때 예외가 발생하면  
    except Exception as e:
        return {"query":"추출된 검색어", "max_calories":None, "min_recommendation":None}

#테스트
userquery = "매운 면 요리인데 칼로리가 최대 1000인 음식"
results_test = analyze_query(userquery)
print(results_test)

{'query': '매운 면 요리', 'max_calories': 1000, 'min_recommendation': None}


In [ ]:
## 메타데이터를 사용해서 유사도 검색
def retrival_docs(query, max_calories=None, min_recommendation=None, k=3):
    filter_conditions={}
    if max_calories is not None:
        #                   키정의     =    $lte(less than or equal) 이하, $lt 미만,
        #                                  $gte(greater than or equal) 이상, $gt 초과
        filter_conditions['calories'] = {"$lte":int(max_calories)}
    if min_recommendation is not None:
        filter_conditions['recommendation'] = {"$gte" : int(min_recommendation)}

    #유사도 검색시 전달할 메타데이터    filter_conditions 있다면 filter_conditions 리턴 아니면 None
    search_filter = filter_conditions if filter_conditions else None
    print(query,search_filter)

    # 이제 유사도 검색 실시
    results = vectorstore.similarity_search(query, k=k, filter= search_filter)

    print(results)
    return results

#테스트
userquery = "매운 면 요리인데 칼로리가 최대 1000인 음식"
results_test = analyze_query(userquery)
query = results_test['query']
max_cal = results_test['max_calories']
min_rec = results_test['min_recommendation']

retrival_docs(query, max_cal, min_rec)  # 노트북에서는 마지막 코드를 출력해서 보여준다
print('test end')

GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 4h25m21.90324448s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content

In [ ]:
## RAG에서 A 단계
def format_docs_with_metadata(docs):
    if not docs:
        return "문서가 없다. 검색된 문서가 없다"
    formatted_list = []

    for doc in docs:
        meta = doc.metadata
        info = f"""
        [메뉴정보]
        - 칼로리:{meta['calories']}
        - 이름 : {meta['name']}
        - 추천수 : {meta['recommendation']}
        [문서 내용]
        {doc.page_content}
         """
        formatted_list.append(info)
        # print(doc)
    return "\n\n".join(formatted_list)

#테스트
userquery = "매운 면 요리인데 칼로리가 최대 1000인 음식"
results_test = analyze_query(userquery)
query = results_test['query']
max_cal = results_test['max_calories']
min_rec = results_test['min_recommendation']

#R
rdocs = retrival_docs(query, max_cal, min_rec)  # 노트북에서는 마지막 코드를 출력해서 보여준다
#A
context = format_docs_with_metadata(rdocs)
print('test end')

매운 면 요리 {'calories': {'$lte': 1000}}
[Document(id='b71addd9-aa85-4f56-b0ce-5ad72051be48', metadata={'calories': 500, 'recommendation': 226, 'name': '쫄면'}, page_content='이름 : 쫄면\n        설명 : 쫄깃한 면을 매콤한 양념에 비빈 음식'), Document(id='db17ab4c-b239-4931-82d1-14d69327365e', metadata={'name': '쫄면', 'calories': 500, 'recommendation': 226}, page_content='이름 : 쫄면\n        설명 : 쫄깃한 면을 매콤한 양념에 비빈 음식'), Document(id='1b66ae00-29b2-4dbd-a896-7d710d7f6b96', metadata={'recommendation': 245, 'calories': 450, 'name': '비빔국수'}, page_content='이름 : 비빔국수\n        설명 : 소면을 매콤새콤한 양념에 비빈 음식')]
page_content='이름 : 쫄면
        설명 : 쫄깃한 면을 매콤한 양념에 비빈 음식' metadata={'calories': 500, 'recommendation': 226, 'name': '쫄면'}
page_content='이름 : 쫄면
        설명 : 쫄깃한 면을 매콤한 양념에 비빈 음식' metadata={'name': '쫄면', 'calories': 500, 'recommendation': 226}
page_content='이름 : 비빔국수
        설명 : 소면을 매콤새콤한 양념에 비빈 음식' metadata={'recommendation': 245, 'calories': 450, 'name': '비빔국수'}
test end


In [18]:
# G
# RAG 최종본
def rag_process(query):
    #1번째 단계는 질의 분석 응답은 json 형태
    results_test = analyze_query(userquery)
    query = results_test['query']
    max_cal = results_test['max_calories']
    min_rec = results_test['min_recommendation']

    #R - 유사도 검색을 함.. 이때 메타데이터도 포함
    rdocs = retrival_docs(query, max_cal, min_rec)  # 노트북에서는 마지막 코드를 출력해서 보여준다

    #A - [메뉴정보] [문서내용]으로 문자열 타입으로 증강
    format_docs_with_metadata(rdocs)
    print('test end')

    #프롬프트
    promt = ChatPromptTemplate.from_template("""
        너는 음식 추천 전문가야. 아래의 참고 정보(메타데이터기반)을 바탕으로 사용자 질문에
        알맞은 메뉴를 1.2 형식으로 대답해줘
        [사용자질문] {query}
        [참고정보(메타데이터기반)] {context}

        [규칙]
        - 제공한 참고 정보의 메타데이터(칼로리, 추천수 등)을 답변에 활용
        - 음식 이름 먼저 이야기 하기
        - 2줄 정도로 간략하게 매력적으로
        - 친절하게 구매하도록 유도
    """)

    chain = promt | llm | StrOutputParser()
    result = chain.invoke({'query' : query, 'context': context})
    return result
# 테스트
userquery = "매운 면 요리인데 칼로리가 최대 1000인 음식"
rag_process(userquery)

매운 면 요리 {'calories': {'$lte': 1000}}
[Document(id='b71addd9-aa85-4f56-b0ce-5ad72051be48', metadata={'name': '쫄면', 'recommendation': 226, 'calories': 500}, page_content='이름 : 쫄면\n        설명 : 쫄깃한 면을 매콤한 양념에 비빈 음식'), Document(id='db17ab4c-b239-4931-82d1-14d69327365e', metadata={'name': '쫄면', 'calories': 500, 'recommendation': 226}, page_content='이름 : 쫄면\n        설명 : 쫄깃한 면을 매콤한 양념에 비빈 음식'), Document(id='1b66ae00-29b2-4dbd-a896-7d710d7f6b96', metadata={'name': '비빔국수', 'calories': 450, 'recommendation': 245}, page_content='이름 : 비빔국수\n        설명 : 소면을 매콤새콤한 양념에 비빈 음식')]
test end


'1. **쫄면**\n쫄깃한 면발과 매콤한 양념의 조합으로 226명에게 검증된 인기 메뉴(500kcal)입니다.'

In [ ]:
## 전체 파이프라인..
#1. 질의
#2. 키워드 분석 (query, 칼로리, 추천수)
#3. 유사도분석(메타데이터 포함)
#4. 종강([내용정보] [문서내용])
#5. 프롬프트
#6. 문장생성